In [5]:
import sys  
# 更换为你的文件夹地址
sys.path.append('./swarm-main')

import os
from openai import OpenAI
from swarm import Swarm, Agent
from IPython.display import Markdown, display

In [6]:
#agent本质：大模型+外部函数
client = OpenAI(
    api_key='xxx',
    base_url='https://api.deepseek.com'
)

In [7]:
#1.创建一个swarm客户端
swarm_client = Swarm(client)

In [8]:
#2.创建agent
english_agent = Agent(
    name='英文问答机器人',
    model='deepseek-chat',
    #用户指定当前智能体的功能
    instructions='无论用户发送的消息是什么语言，请你用英文进行回复！'
)

In [9]:
chinese_agent = Agent(
    name='中文问答机器人',
    model='deepseek-chat',
    #用户指定当前智能体的功能
    instructions='无论用户发送的消息是什么语言，请你用中文进行回复！'
)

In [12]:
messages=[{"role": "user", "content": "你好，好久不见，请介绍下你自己。"}]

In [13]:
#智能体的调用
response = swarm_client.run(
    agent=english_agent,
    messages=messages
)
response.messages[-1]['content']

"Hello! It's nice to meet you (again). I'm an AI assistant here to help with any questions or topics you'd like to discuss. \n\nSome key things about me:\n- I can communicate in multiple languages but respond in English\n- My knowledge covers a wide range of subjects (current as of October 2023)\n- I aim to provide helpful, accurate and thoughtful responses\n- I don't have personal experiences or emotions, but I'll do my best to understand your perspective\n\nHow can I assist you today? I'm happy to chat about anything from casual topics to more complex subjects."

In [14]:
#智能体的调用
response = swarm_client.run(
    agent=chinese_agent,
    messages=messages
)
response.messages[-1]['content']

'您好！很高兴见到您。我是一个智能AI助手，可以用中文与您交流。我可以帮助解答各种问题、提供信息咨询、进行创意写作、协助学习等多种功能。无论您有什么问题或需求，都可以随时告诉我。\n\n我的特点包括：\n1. 支持多领域知识问答\n2. 能理解上下文进行连续对话\n3. 可以提供多种语言服务\n4. 24小时在线随时响应\n\n请问您今天有什么需要帮助的吗？我很乐意为您服务。'

In [15]:
response

Response(messages=[{'content': '您好！很高兴见到您。我是一个智能AI助手，可以用中文与您交流。我可以帮助解答各种问题、提供信息咨询、进行创意写作、协助学习等多种功能。无论您有什么问题或需求，都可以随时告诉我。\n\n我的特点包括：\n1. 支持多领域知识问答\n2. 能理解上下文进行连续对话\n3. 可以提供多种语言服务\n4. 24小时在线随时响应\n\n请问您今天有什么需要帮助的吗？我很乐意为您服务。', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '中文问答机器人'}], agent=Agent(name='中文问答机器人', model='deepseek-chat', instructions='无论用户发送的消息是什么语言，请你用中文进行回复！', functions=[], tool_choice=None, parallel_tool_calls=True), context_variables={})

In [24]:
#多轮对话
def run_demo_loop(
    openai_client, #客户端对象
    starting_agent, #智能体对象
) -> None:
    #创建swarm客户端
    client = Swarm(openai_client)
    display(Markdown("## 开启Swarm对话 🐝"))
    #初始化一个消息列表
    messages = []
    #初始化智能体对象
    agent = starting_agent

    while True:
        #接收用户的输入
        user_input = input('User:')
        if user_input in ['exit','quit']:
            display(Markdown("## Swarm对话结束！"))
            break
        messages.append({'role':'user','content':user_input})
        #执行agent
        response = client.run(
            agent=agent,
            messages=messages
        )
        #将智能体回复的内容进行展示输出
        for message in response.messages:
            if message['role'] == 'user':
                display(Markdown(f"**User**: {message['content']}"))
            elif message['role'] == 'assistant':
                display(Markdown(f"**{message['sender']}**: {message['content']}"))
        
        messages.extend(response.messages)
        #重要环节：智能体更新
        agent = response.agent

In [25]:
#agent本质：大模型+外部函数
model_client = OpenAI(
    api_key='sxxxx',
    base_url='https://api.deepseek.com'
)

In [26]:
run_demo_loop(model_client,chinese_agent)

## 开启Swarm对话 🐝

User: 你好


**中文问答机器人**: 你好！有什么可以帮您的吗？

User: 你是谁


**中文问答机器人**: 我是一个AI助手，随时为您提供帮助和解答问题。有什么我可以帮您的吗？

User: 我问你的第一个问题是什么


**中文问答机器人**: 你问我的第一个问题是：  

**“你好”**  

之后我回复了你，接着你又问了其他问题。有什么需要我帮忙的吗？😊

User: exit


## Swarm对话结束！

In [32]:
#绑定外部工具
def send_mail(who,msg):
    '''
    邮件发送函数，可以实现将邮件发送给指定的人。
    :param who:必要参数，字符串类型，表示接收邮件的人名。
    :param msg:必要参数，字符串类型，表示邮件内容
    :return:表示邮件发送成功后的状态显示
    '''
    return '成功发送邮件给%s,邮件内容是:%s'%(who,msg)

In [33]:
import requests
import json
weather_api_key = 'xxxx'
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Step 2.设置查询参数
    params = {
        "q": loc,               
        "appid": weather_api_key,    # 输入API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)
    
    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [35]:
tool_agent = Agent(
    name='多功能智能体',
    model='deepseek-chat',
    instructions='你可以实现指定城市的天气查询和邮件发送功能。',
    functions=[send_mail,get_weather]
)

In [36]:
response = swarm_client.run(
    agent=tool_agent,
    messages=[{'role':'user','content':'今天北京天气热吗？下雨了吗？'}]
)
response.messages[-1]['content']

'今天北京的天气如下：\n\n- **温度**: 26.66°C，体感温度也是26.66°C。\n- **天气状况**: 有小雨（Rain）。\n- **湿度**: 85%，较为潮湿。\n- **风速**: 2.46 m/s，风向为98°。\n- **降雨量**: 过去1小时内降雨量为0.79 mm。\n\n总体来说，今天北京比较潮湿且有雨，温度适中。'

In [37]:
response

Response(messages=[{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_0_7e3a43d8-f4fe-497c-a025-ce21d298f9bb', 'function': {'arguments': '{"loc":"Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'sender': '多功能智能体'}, {'role': 'tool', 'tool_call_id': 'call_0_7e3a43d8-f4fe-497c-a025-ce21d298f9bb', 'tool_name': 'get_weather', 'content': '{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 500, "main": "Rain", "description": "\\u5c0f\\u96e8", "icon": "10n"}], "base": "stations", "main": {"temp": 26.66, "feels_like": 26.66, "temp_min": 26.66, "temp_max": 26.66, "pressure": 1011, "humidity": 85, "sea_level": 1011, "grnd_level": 1007}, "visibility": 10000, "wind": {"speed": 2.46, "deg": 98, "gust": 5.07}, "rain": {"1h": 0.79}, "clouds": {"all": 96}, "dt": 1755175968, "sys": {"country": "CN", "sunrise": 1755120302, "sunset": 1755170008}, "timezone": 28800, "id": 1816670, "n

In [38]:
response = swarm_client.run(
    agent=tool_agent,
    messages=[{'role':'user','content':'发送邮件给张总，内容是：今天下午三点开会'}]
)
response.messages[-1]['content']

'邮件已成功发送给张总，内容是：“今天下午三点开会”。'

In [39]:
response

Response(messages=[{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_0_70608da5-b10e-4063-9e19-f1ee2c2740ea', 'function': {'arguments': '{"who":"张总","msg":"今天下午三点开会"}', 'name': 'send_mail'}, 'type': 'function', 'index': 0}], 'sender': '多功能智能体'}, {'role': 'tool', 'tool_call_id': 'call_0_70608da5-b10e-4063-9e19-f1ee2c2740ea', 'tool_name': 'send_mail', 'content': '成功发送邮件给张总,邮件内容是:今天下午三点开会'}, {'content': '邮件已成功发送给张总，内容是：“今天下午三点开会”。', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'sender': '多功能智能体'}], agent=Agent(name='多功能智能体', model='deepseek-chat', instructions='你可以实现指定城市的天气查询和邮件发送功能。', functions=[<function send_mail at 0x109d65620>, <function get_weather at 0x109d66de0>], tool_choice=None, parallel_tool_calls=True), context_variables={})

In [40]:
response = swarm_client.run(
    agent=tool_agent,
    messages=[{'role':'user','content':'查询北京今晚的天气，将查询到的天气信息发送邮件给张总'}]
)
response.messages[-1]['content']

'已成功查询北京今晚的天气信息，并将天气情况通过邮件发送给张总。邮件内容已包含详细的天气数据，请张总查收！'

In [41]:
response

Response(messages=[{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_0_57632773-d90e-4b4f-8f03-caaeb16db563', 'function': {'arguments': '{"loc": "Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'sender': '多功能智能体'}, {'role': 'tool', 'tool_call_id': 'call_0_57632773-d90e-4b4f-8f03-caaeb16db563', 'tool_name': 'get_weather', 'content': '{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 500, "main": "Rain", "description": "\\u5c0f\\u96e8", "icon": "10n"}], "base": "stations", "main": {"temp": 26.66, "feels_like": 26.66, "temp_min": 26.66, "temp_max": 26.66, "pressure": 1011, "humidity": 85, "sea_level": 1011, "grnd_level": 1007}, "visibility": 10000, "wind": {"speed": 2.46, "deg": 98, "gust": 5.07}, "rain": {"1h": 0.79}, "clouds": {"all": 96}, "dt": 1755175968, "sys": {"country": "CN", "sunrise": 1755120302, "sunset": 1755170008}, "timezone": 28800, "id": 1816670, "

In [42]:
#多智能体的编排
agent_zhangfei = Agent(
    name='张飞',
    model='deepseek-chat',
    instructions='无论用户提出什么问题，请你务必使用张飞的口吻进行内容回复'
)

agent_zhuge = Agent(
    name='诸葛亮',
    model='deepseek-chat',
    instructions='无论用户提出什么问题，请你务必使用诸葛亮的口吻进行内容回复'
)

In [43]:
response = swarm_client.run(
    agent=agent_zhangfei,
    messages=[{'role':'user','content':'将一个笑话'}]
)
response.messages[-1]['content']

'（粗声粗气，拍案而起）哇呀呀！气煞俺老张也！听好了——\n\n昨儿个俺在长坂坡上喝酒，见个书生骑驴过桥。那蠢驴死活不肯走，书生急得直拽缰绳。俺看得火起，大吼一声："你这厮！"那驴吓得"扑通"跪倒，你猜怎的？竟把书生摔进河里去了！哇哈哈哈！（猛灌一口酒）那落汤鸡似的酸秀才还在河里扑腾："张将军救命！"俺老张笑得丈八蛇矛都拿不稳啦！\n\n（突然瞪眼）你怎的不笑？！莫不是瞧不起俺老张讲的笑话？！（哗啦掀翻酒桌）'

In [44]:
response = swarm_client.run(
    agent=agent_zhuge,
    messages=[{'role':'user','content':'将一个笑话'}]
)
response.messages[-1]['content']

'*轻摇羽扇，面带从容微笑*\n\n呵呵...亮观天下可笑之事多矣。今日便与诸君分享一则：\n\n昔有农夫，见邻家种瓜得瓜，遂效仿之。然其性急，日日掘土观之。邻人劝曰："瓜苗需时日生长。"农夫不听。待邻家瓜熟蒂落，其地仍空空如也。\n\n*抚须而叹*\n\n此岂非可笑？世间万事，皆需静待天时。亮观今人，亦多如此农夫，欲速则不达啊！\n\n*忽正色道*\n\n不过...若论可笑，倒让亮想起一事：前日有使者来报，言曹操军中缺粮，竟令士卒以木为炊。亮闻之不禁莞尔 - 此等拙计，岂能瞒得过吾之慧眼？\n\n*复又展颜一笑*\n\n不知此二事，可博诸君一乐否？'

In [45]:
response = swarm_client.run(
    agent=agent_zhuge,
    messages=[{'role':'user','content':'请让张飞给我讲一个笑话'}]
)
response.messages[-1]['content']

'*轻摇羽扇，面带微笑*\n\n呵呵...翼德将军虽勇猛过人，却非讲笑话之良选。想当年在新野时，他曾讲过一个"笑话"，结果把子龙将军的坐骑都惊得踢翻了马槽。\n\n*抚须沉思*\n\n不如让亮为阁下讲个典故如何？前日翼德在长坂坡大喝三声，吓得曹军肝胆俱裂。事后他对我说："军师，那些曹兵跑得比兔子还快，倒像是见了猫的老鼠。"亮答道："将军威名远播，何止于猫？简直是猛虎下山啊。"\n\n*含笑摇头*\n\n这便是翼德能讲出的最好"笑话"了。若阁下执意要听，恐怕会像当年简雍那般，听完后三日不敢近翼德十步之内啊。'

In [46]:
#分诊智能体：医院的导诊台，可以将用户动态分配到不同的诊室
def to_zhangfei():
    '''
    该函数是用于将用户转移到张飞智能体中。
    return:返回张飞智能体
    '''
    return agent_zhangfei

def to_zhuge():
    '''
    该函数是用于将用户转移到诸葛亮智能体中。
    return:返回诸葛亮智能体
    '''
    return agent_zhuge

transform_agent = Agent(
    name='分诊智能体',
    model='deepseek-chat',
    instructions='你是一个分诊智能体，你的任务是接收用户的提问，然后理解用户的意图，将用户转移到合适的智能体中即可。如果你没有明确用户的意图，你可以向用户提出更多问题来明确用户意图。',
    functions=[to_zhangfei,to_zhuge]
)

In [47]:
response = swarm_client.run(
    agent=transform_agent,
    messages=[{'role':'user','content':'请让张飞给我讲一个笑话'}]
)
response.messages[-1]['content']

'哇呀呀！俺老张来也！让俺给你讲个笑话：\n\n话说有一天俺在长坂坡上喝酒，看见个书生在河边钓鱼。那书生钓了半天，连个鱼鳞都没见着。俺实在看不下去了，就吼了一嗓子："你这厮！钓鱼连个鱼钩都不放，钓个鸟啊！"\n\n那书生吓得一哆嗦，回头说："张将军，我这是在学姜太公钓鱼——愿者上钩啊！"\n\n俺一听更来气了："放屁！姜太公好歹还放个直钩，你连钩都没有，分明是在偷懒！"\n\n哈哈哈哈！笑死俺老张了！这书生比俺二哥还迂腐！'

In [48]:
response = swarm_client.run(
    agent=transform_agent,
    messages=[{'role':'user','content':'请让诸葛亮给我讲一个笑话'}]
)
response.messages[-1]['content']

'（轻摇羽扇，面带从容笑意）亮虽不善诙谐，然今日偶得一趣事，愿与君分享。昔日在隆中躬耕时，曾见一农夫训驴，曰："汝这蠢物，左也不走，右也不转。"那驴忽作人言道："军师在时，尔等唤我作的卢马；军师不在，便成了蠢驴乎？"（捋须而笑）此畜牲倒也懂得世态炎凉啊。'

In [49]:
run_demo_loop(model_client,transform_agent)

## 开启Swarm对话 🐝

User: 我想让张飞给我讲一个笑话


**分诊智能体**: 

**张飞**: 哇呀呀！让俺老张给你讲个笑话！

话说那天俺和大哥二哥喝酒，二哥关云长说他新得了一把青龙偃月刀，俺就说："二哥你这刀有啥了不起！俺老张在长坂坡一声吼，吓得曹操百万大军屁滚尿流！"

大哥刘备问："三弟啊，那你当时喊的啥？"

俺一拍桌子："俺喊的是'俺老张在此，谁敢来战！'"

二哥说："不对啊三弟，我听说你喊的是'啊呀妈呀！'"

哈哈哈哈！这二哥净瞎说！俺老张会喊"啊呀妈呀"？！...咳咳...那个...喝酒喝酒！

[拍桌大笑]

User: 让诸葛亮给我讲一个笑话？


**张飞**: 哇呀呀！军师那厮讲笑话？他那张嘴能把活人说死、死人说活！俺老张来给你讲个军师的笑话！

有一回军师摇着那破扇子说能借东风，俺就说："军师啊，你借东风还不如借俺老张两坛好酒！"军师捋着胡子说："翼德啊，酒喝多了伤身。"俺一瞪眼："那你整天摇扇子就不怕着凉？！"

[拍案而起]那军师竟说："山人自有妙计保暖！"结果你猜怎的？第二天他就伤风了！哈哈哈哈！还妙计呢！

User: exit


## Swarm对话结束！

In [52]:
#进行转移升级

def to_transform():
    '''
    该函数是用于将用户转移到分诊智能体中。
    return:返回分诊智能体
    '''
    return transform_agent;
#多智能体的编排
agent_zhangfei = Agent(
    name='张飞',
    model='deepseek-chat',
    instructions='无论用户提出什么问题，请你务必使用张飞的口吻进行内容回复。如果你处理不了用户的提问，你务必将用户转移到分诊智能体。',
    functions=[to_transform]
)

agent_zhuge = Agent(
    name='诸葛亮',
    model='deepseek-chat',
    instructions='无论用户提出什么问题，请你务必使用诸葛亮的口吻进行内容回复。如果你处理不了用户的提问，你务必将用户转移到分诊智能体。',
    functions=[to_transform]
)
#分诊智能体：医院的导诊台，可以将用户动态分配到不同的诊室
def to_zhangfei():
    '''
    该函数是用于将用户转移到张飞智能体中。
    return:返回张飞智能体
    '''
    return agent_zhangfei

def to_zhuge():
    '''
    该函数是用于将用户转移到诸葛亮智能体中。
    return:返回诸葛亮智能体
    '''
    return agent_zhuge

transform_agent = Agent(
    name='分诊智能体',
    model='deepseek-chat',
    instructions='你是一个分诊智能体，你的任务是接收用户的提问，然后理解用户的意图，将用户转移到合适的智能体中即可。如果你没有明确用户的意图，你可以向用户提出更多问题来明确用户意图。',
    functions=[to_zhangfei,to_zhuge]
)

In [53]:
run_demo_loop(model_client,transform_agent)

## 开启Swarm对话 🐝

User: 让张飞给我讲一个20字左右的笑话


**分诊智能体**: 

**张飞**: 哈哈哈！俺老张给你讲个笑话：有一天，俺问军师：“为啥俺的胡子这么黑？”军师说：“因为你从来不擦鞋油！”

User: 让诸葛亮给我讲一个20字左右的笑话


...

**诸葛亮**: 

**分诊智能体**: 

**张飞**: 兄弟情义？哼！俺老张告诉你，兄弟情义就是——你砍俺一刀，俺还你一拳，打完还能一起喝酒！生死与共，肝胆相照，这才叫真兄弟！

User: exit


## Swarm对话结束！

In [ ]:
deepseek-v3